In [1]:
"""
Kaggle PS S6E4 — Predicting Irrigation Need  v3
=================================================
XGB (multi-seed) + CatBoost + LightGBM  |  10-Fold
Pairwise combos, orig priors, log-space bias tuning.

v2 → v3 changes:
  - Added LightGBM as 3rd model
  - XGB: LR 0.05→0.02, ES 150→300, 2 seeds per fold
  - 10-fold instead of 5
  - 3-model blend weight search
"""

import gc
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgbm
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
import warnings

warnings.filterwarnings("ignore")

TARGET = "Irrigation_Need"
INTERNAL_ORDER = ["Low", "Medium", "High"]
PUBLIC_ORDER   = ["High", "Low", "Medium"]

NUMS = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
]
CATS = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]
ALL_SOURCE = NUMS + CATS

ORIG_ROW_WEIGHT = 0.35
N_FOLDS = 10
SEED = 42
XGB_SEEDS = [42, 43]  # multi-seed ensemble

# ═══════════════════════════════════════════════════════════════════════
# 1. LOAD
# ═══════════════════════════════════════════════════════════════════════
DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e4")
ORIG_DIR = Path("/kaggle/input/irrigation-water-requirement-prediction-dataset")

train = pd.read_csv(DATA_DIR / "train.csv").set_index("id")
test  = pd.read_csv(DATA_DIR / "test.csv").set_index("id")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

try:
    orig = pd.read_csv(next(ORIG_DIR.rglob("*.csv")))
except StopIteration:
    orig = pd.DataFrame(columns=train.columns)

print(f"Train: {train.shape}, Test: {test.shape}, Orig: {orig.shape}")

# ═══════════════════════════════════════════════════════════════════════
# 2. HELPERS
# ═══════════════════════════════════════════════════════════════════════
def bal_acc_xgb_metric():
    def _m(y_true, y_pred):
        return balanced_accuracy_score(
            y_true.astype(int), np.argmax(y_pred.reshape(-1, 3), axis=1)
        )
    _m.__name__ = "bal_acc"
    return _m


def public_preds(proba, bias):
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)


def tune_bias(proba, y_true):
    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, public_preds(proba, best_bias))
    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    c = best_bias.copy()
                    c[ci] += d * step
                    s = balanced_accuracy_score(y_true, public_preds(proba, c))
                    if s > best_score + 1e-8:
                        best_bias, best_score, improved = c, s, True
    return best_bias, best_score


def reorder_to_public(proba):
    src = {l: i for i, l in enumerate(INTERNAL_ORDER)}
    return proba[:, [src[l] for l in PUBLIC_ORDER]]

# ═══════════════════════════════════════════════════════════════════════
# 3. FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════
def factorize_joint(tr, te, og):
    for col in CATS:
        combined = pd.concat([tr[col], te[col], og[col]], ignore_index=True)
        codes, _ = pd.factorize(combined)
        tr[col] = pd.Series(codes[:len(tr)], index=tr.index).astype("int32").astype("category")
        te[col] = pd.Series(codes[len(tr):len(tr)+len(te)], index=te.index).astype("int32").astype("category")
        og[col] = pd.Series(codes[len(tr)+len(te):], index=og.index).astype("int32").astype("category")


def build_pair_columns(tr, te, og):
    te_cols = []
    total = len(tr) + len(te) + len(og)
    for left, right in combinations(ALL_SOURCE, 2):
        name = f"{left}__{right}"
        tr[name] = tr[left].astype(str) + "_" + tr[right].astype(str)
        te[name] = te[left].astype(str) + "_" + te[right].astype(str)
        og[name] = og[left].astype(str) + "_" + og[right].astype(str)
        combined = pd.concat([tr[name], te[name], og[name]], ignore_index=True)
        codes, _ = pd.factorize(combined)
        if pd.Series(codes).nunique() > total // 2:
            tr.drop(columns=[name], inplace=True)
            te.drop(columns=[name], inplace=True)
            og.drop(columns=[name], inplace=True)
            continue
        tr[name] = codes[:len(tr)].astype("int32")
        te[name] = codes[len(tr):len(tr)+len(te)].astype("int32")
        og[name] = codes[len(tr)+len(te):].astype("int32")
        te_cols.append(name)
    return te_cols


def add_orig_priors(tr, te, og):
    created = []
    for col in CATS + NUMS:
        mapping = og.groupby(col, observed=False)[TARGET].mean().astype("float32")
        name = f"TE_ORIG_{col}"
        tr[name] = tr[col].astype(object).map(mapping).astype("float32").fillna(0.5)
        te[name] = te[col].astype(object).map(mapping).astype("float32").fillna(0.5)
        og[name] = og[col].astype(object).map(mapping).astype("float32").fillna(0.5)
        created.append(name)
    return created


def add_domain_features(*dfs):
    for d in dfs:
        d["moist_rain"]    = d["Soil_Moisture"] / (d["Rainfall_mm"] + 1)
        d["moist_temp"]    = d["Soil_Moisture"] / (d["Temperature_C"] + 1)
        d["moist_wind"]    = d["Soil_Moisture"] / (d["Wind_Speed_kmh"] + 1)
        d["ET_proxy"]      = (d["Temperature_C"] * d["Wind_Speed_kmh"] * d["Sunlight_Hours"]) / (d["Humidity"] + 1)
        d["heat_stress"]   = d["Temperature_C"] * d["Sunlight_Hours"]
        d["drying_force"]  = d["Wind_Speed_kmh"] * d["Temperature_C"] / (d["Humidity"] + 1)
        d["water_supply"]  = d["Rainfall_mm"] + d["Previous_Irrigation_mm"]
        d["water_deficit"] = d["Soil_Moisture"] - d["water_supply"] * 0.1
        d["soil_quality"]  = d["Organic_Carbon"] / (d["Electrical_Conductivity"] + 0.1)
        d["moist_x_wind"]  = d["Soil_Moisture"] * d["Wind_Speed_kmh"]
        d["moist_x_temp"]  = d["Soil_Moisture"] * d["Temperature_C"]
        d["wind_x_temp"]   = d["Wind_Speed_kmh"] * d["Temperature_C"]
        d["moisture_sq"]   = d["Soil_Moisture"] ** 2
        d["wind_sq"]       = d["Wind_Speed_kmh"] ** 2
        d["temp_sq"]       = d["Temperature_C"] ** 2


print("Encoding targets...")
target_map = {l: i for i, l in enumerate(INTERNAL_ORDER)}
train[TARGET] = train[TARGET].map(target_map)
orig[TARGET]  = orig[TARGET].map(target_map)
y_internal = train[TARGET].to_numpy(dtype=np.int64)

public_map = {l: i for i, l in enumerate(PUBLIC_ORDER)}
i2p = {target_map[l]: public_map[l] for l in INTERNAL_ORDER}
y_public = np.array([i2p[v] for v in y_internal], dtype=np.int64)

print("Factorizing...")
factorize_joint(train, test, orig)

print("Pairwise combos...")
te_columns = build_pair_columns(train, test, orig)
print(f"  Kept {len(te_columns)} pairs")

print("Orig priors...")
add_orig_priors(train, test, orig)

print("Domain features...")
add_domain_features(train, test, orig)

feature_columns = [c for c in train.columns if c != TARGET]
print(f"Total features: {len(feature_columns)}")

# ═══════════════════════════════════════════════════════════════════════
# 4. PER-FOLD TRAINING
# ═══════════════════════════════════════════════════════════════════════
def build_frames(tr_fold, va_fold, te_df, og_df, feat_cols, te_cols, y_tr, seed):
    base_cols = [c for c in feat_cols if c not in te_cols]
    encoder = TargetEncoder(target_type="multiclass", cv=5, random_state=seed)

    model_raw    = pd.concat([tr_fold[feat_cols], og_df[feat_cols]], ignore_index=True)
    model_target = np.concatenate([y_tr, og_df[TARGET].to_numpy(dtype=np.int64)])
    te_raw       = pd.concat([tr_fold[te_cols], og_df[te_cols]], ignore_index=True)

    def _tf(a): return pd.DataFrame(a, columns=[f"te_{i}" for i in range(a.shape[1])])

    enc_tr = _tf(encoder.fit_transform(te_raw, model_target))
    enc_va = _tf(encoder.transform(va_fold[te_cols]))
    enc_te = _tf(encoder.transform(te_df[te_cols]))

    x_tr = pd.concat([enc_tr, model_raw[base_cols].reset_index(drop=True)], axis=1)
    x_va = pd.concat([enc_va, va_fold[base_cols].reset_index(drop=True)], axis=1)
    x_te = pd.concat([enc_te, te_df[base_cols].reset_index(drop=True)], axis=1)

    sw = compute_sample_weight("balanced", model_target).astype(np.float32)
    sw[len(tr_fold):] *= ORIG_ROW_WEIGHT

    cat_base = [c for c in CATS if c in x_tr.columns]
    for f in (x_tr, x_va, x_te):
        for c in cat_base:
            f[c] = f[c].astype("category")

    return x_tr, model_target, x_va, x_te, cat_base, sw


splitter = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
oof_xgb = np.zeros((len(train), 3), dtype=np.float64)
oof_cb  = np.zeros((len(train), 3), dtype=np.float64)
oof_lgb = np.zeros((len(train), 3), dtype=np.float64)
tst_xgb_parts, tst_cb_parts, tst_lgb_parts = [], [], []

print(f"\n{'='*70}")
print(f"  {N_FOLDS}-FOLD  |  XGB ({len(XGB_SEEDS)} seeds) + CatBoost + LightGBM")
print(f"{'='*70}\n")

for fold, (ti, vi) in enumerate(splitter.split(np.zeros(len(train)), y_internal), 1):
    print(f"── Fold {fold}/{N_FOLDS} ──")
    tr_f, va_f = train.iloc[ti].copy(), train.iloc[vi].copy()
    y_tr = tr_f[TARGET].to_numpy(dtype=np.int64)
    y_va = va_f[TARGET].to_numpy(dtype=np.int64)

    x_tr, y_m, x_va, x_te, cat_b, sw = build_frames(
        tr_f, va_f, test, orig, feature_columns, te_columns, y_tr, SEED + fold
    )

    # ── XGBoost (multi-seed) ────────────────────────────────────────
    va_xgb_seeds, te_xgb_seeds = [], []
    for s in XGB_SEEDS:
        m = xgb.XGBClassifier(
            max_depth=6, subsample=0.8, colsample_bytree=0.8,
            n_estimators=10000, objective="multi:softprob",
            learning_rate=0.02, min_child_weight=3, max_bin=1024,
            eval_metric=bal_acc_xgb_metric(),
            callbacks=[xgb.callback.EarlyStopping(
                rounds=300, metric_name="bal_acc",
                maximize=True, save_best=True,
            )],
            random_state=s, enable_categorical=True,
            device="cuda", tree_method="hist", n_jobs=-1,
        )
        m.fit(x_tr, y_m, eval_set=[(x_va, y_va)], sample_weight=sw, verbose=500)
        va_xgb_seeds.append(m.predict_proba(x_va))
        te_xgb_seeds.append(m.predict_proba(x_te))
        del m; gc.collect()

    va_xgb_avg = np.mean(va_xgb_seeds, axis=0)
    te_xgb_avg = np.mean(te_xgb_seeds, axis=0)
    oof_xgb[vi] = va_xgb_avg
    tst_xgb_parts.append(te_xgb_avg.astype(np.float32))

    # ── CatBoost ────────────────────────────────────────────────────
    cat_idx = [x_tr.columns.get_loc(c) for c in cat_b]
    cb_m = CatBoostClassifier(
        loss_function="MultiClass", eval_metric="MultiClass",
        iterations=1500, learning_rate=0.04, depth=8, l2_leaf_reg=4.0,
        random_seed=SEED + fold, task_type="GPU", devices="0", verbose=500,
    )
    cb_m.fit(
        Pool(x_tr, y_m, cat_features=cat_idx, weight=sw),
        eval_set=Pool(x_va, y_va, cat_features=cat_idx),
        use_best_model=True,
    )
    oof_cb[vi] = cb_m.predict_proba(Pool(x_va, cat_features=cat_idx))
    tst_cb_parts.append(cb_m.predict_proba(Pool(x_te, cat_features=cat_idx)).astype(np.float32))
    del cb_m; gc.collect()

    # ── LightGBM ────────────────────────────────────────────────────
    lgb_m = lgbm.LGBMClassifier(
        objective="multiclass", num_class=3, metric="multi_logloss",
        learning_rate=0.04, num_leaves=127, min_child_samples=50,
        subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,
        class_weight="balanced", n_estimators=3000,
        random_state=SEED + fold, verbose=-1, n_jobs=-1,
    )
    lgb_m.fit(
        x_tr, y_m, sample_weight=sw,
        eval_set=[(x_va, y_va)],
        callbacks=[lgbm.early_stopping(150, verbose=False), lgbm.log_evaluation(0)],
    )
    oof_lgb[vi] = lgb_m.predict_proba(x_va)
    tst_lgb_parts.append(lgb_m.predict_proba(x_te).astype(np.float32))
    del lgb_m; gc.collect()

    ba_x = balanced_accuracy_score(y_va, va_xgb_avg.argmax(1))
    ba_c = balanced_accuracy_score(y_va, oof_cb[vi].argmax(1))
    ba_l = balanced_accuracy_score(y_va, oof_lgb[vi].argmax(1))
    print(f"  XGB={ba_x:.5f}  CB={ba_c:.5f}  LGB={ba_l:.5f}\n")

    del x_tr, x_va, x_te, tr_f, va_f
    gc.collect()

# ═══════════════════════════════════════════════════════════════════════
# 5. 3-MODEL BLEND + BIAS TUNE
# ═══════════════════════════════════════════════════════════════════════
mean_tst_xgb = np.mean(np.stack(tst_xgb_parts), axis=0)
mean_tst_cb  = np.mean(np.stack(tst_cb_parts), axis=0)
mean_tst_lgb = np.mean(np.stack(tst_lgb_parts), axis=0)

print(f"\n{'='*70}")
print("  INDIVIDUAL OOF SCORES")
print(f"{'='*70}")
for nm, o in [("XGB", oof_xgb), ("CB", oof_cb), ("LGB", oof_lgb)]:
    print(f"  {nm}: {balanced_accuracy_score(y_internal, o.argmax(1)):.5f}")

# Grid search 3-way blend
print("\nSearching blend weights...")
best_sc, best_w = 0, (1/3, 1/3, 1/3)
for w1 in np.arange(0.10, 0.80, 0.05):
    for w2 in np.arange(0.05, 0.80 - w1, 0.05):
        w3 = round(1.0 - w1 - w2, 2)
        if w3 < 0.05:
            continue
        bl = w1 * oof_xgb + w2 * oof_cb + w3 * oof_lgb
        s = balanced_accuracy_score(y_internal, bl.argmax(1))
        if s > best_sc + 1e-8:
            best_sc = s
            best_w = (round(w1, 2), round(w2, 2), w3)

print(f"  Weights: XGB={best_w[0]}, CB={best_w[1]}, LGB={best_w[2]}")
print(f"  Blended BA: {best_sc:.5f}")

hybrid_oof  = best_w[0]*oof_xgb + best_w[1]*oof_cb + best_w[2]*oof_lgb
hybrid_test = best_w[0]*mean_tst_xgb + best_w[1]*mean_tst_cb + best_w[2]*mean_tst_lgb

oof_pub  = reorder_to_public(hybrid_oof)
test_pub = reorder_to_public(hybrid_test)

print("\nTuning bias...")
bias, bias_score = tune_bias(oof_pub, y_public)
print(f"  Bias: {bias}")
print(f"  Final OOF BA: {bias_score:.5f}")

# ═══════════════════════════════════════════════════════════════════════
# 6. SUBMISSION
# ═══════════════════════════════════════════════════════════════════════
decode = np.array(PUBLIC_ORDER)
submission = sample_sub.copy()
submission[TARGET] = decode[public_preds(test_pub, bias)]
submission.to_csv("submission.csv", index=False)

print(f"\n{'='*70}")
print(f"  submission.csv ({len(submission)} rows)")
print(f"{'='*70}")
print(submission[TARGET].value_counts().to_string())

np.save("oof_xgb.npy", oof_xgb)
np.save("oof_cb.npy", oof_cb)
np.save("oof_lgb.npy", oof_lgb)
np.save("test_blend.npy", test_pub)
print("\nDone!")

Train: (630000, 20), Test: (270000, 19), Orig: (10000, 20)
Encoding targets...
Factorizing...
Pairwise combos...
  Kept 135 pairs
Orig priors...
Domain features...
Total features: 188

  10-FOLD  |  XGB (2 seeds) + CatBoost + LightGBM

── Fold 1/10 ──
[0]	validation_0-mlogloss:1.07277	validation_0-bal_acc:0.95926
[500]	validation_0-mlogloss:0.05655	validation_0-bal_acc:0.97647
[828]	validation_0-mlogloss:0.05205	validation_0-bal_acc:0.97591
[0]	validation_0-mlogloss:1.07282	validation_0-bal_acc:0.96069
[500]	validation_0-mlogloss:0.05643	validation_0-bal_acc:0.97623
[1000]	validation_0-mlogloss:0.05092	validation_0-bal_acc:0.97572
[1150]	validation_0-mlogloss:0.05033	validation_0-bal_acc:0.97573
0:	learn: 1.0289515	test: 1.0285224	best: 1.0285224 (0)	total: 269ms	remaining: 6m 43s
500:	learn: 0.0566575	test: 0.0610622	best: 0.0610622 (500)	total: 38.6s	remaining: 1m 17s
1000:	learn: 0.0472313	test: 0.0567890	best: 0.0567890 (1000)	total: 1m 12s	remaining: 36.1s
1499:	learn: 0.0415649	t